# Phase 1 — Descarga ERA5 (variables meteorológicas)

**Entorno**: Local  
**Tiempo estimado**: 20–40 min (el CDS procesa en cola en sus servidores)  
**Output**: archivos NetCDF en `data/era5/`

Descarga variables ERA5 Single-Levels reanalysis sobre el AOI de Corrientes, dic 2021 – feb 2022.

Variables descargadas:
- `u10` — componente zonal del viento a 10 m (m/s)
- `v10` — componente meridional del viento a 10 m (m/s)
- `t2m` — temperatura a 2 m (K)
- `d2m` — temperatura de punto de rocío a 2 m (K → para humedad relativa)
- `tp`  — precipitación total acumulada (m)

Resolución nativa ERA5: 0.25° × 0.25° (~28 km). Se remuestreará a la resolución de Sentinel-2 en Phase 2.

## 1. Imports

In [ ]:
import cdsapi
from pathlib import Path

DATA_DIR = Path("..") / "data" / "era5"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Lee automáticamente C:\Users\<user>\.cdsapirc
client = cdsapi.Client(quiet=False)
print(f"CDS client OK — {client.url}")
print(f"Output dir: {DATA_DIR.resolve()}")

## 2. Parámetros del AOI

In [ ]:
# ERA5 area: [N, W, S, E] — orden diferente al BBOX estándar
# Añadimos 0.5° de margen para evitar artefactos en el borde al reproyectar
ERA5_AREA = [-26.0, -60.0, -29.5, -55.5]  # [N, W, S, E]

# Meses a descargar
REQUESTS = [
    {"year": "2021", "month": "12", "label": "2021-12"},
    {"year": "2022", "month": "01", "label": "2022-01"},
    {"year": "2022", "month": "02", "label": "2022-02"},
]

# Variables ERA5
VARIABLES = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_temperature",
    "2m_dewpoint_temperature",
    "total_precipitation",
]

# Todos los días del mes, hora 12:00 UTC (mediodía local ~ hora de paso Sentinel-2)
DAYS = [str(d).zfill(2) for d in range(1, 32)]
TIMES = ["12:00"]

print(f"AOI ERA5 [N,W,S,E]: {ERA5_AREA}")
print(f"Variables: {VARIABLES}")
print(f"Meses: {[r['label'] for r in REQUESTS]}")

## 3. Descarga — un archivo NetCDF por mes

El CDS encola el request en sus servidores. El tiempo de espera varía entre 2 y 30 minutos
dependiendo de la carga del sistema. El cliente `cdsapi` espera automáticamente y descarga
cuando el archivo está listo.

In [ ]:
import zipfile
import xarray as xr
import tempfile

def retrieve_era5(client, request_params, output_path):
    """
    Descarga ERA5 del CDS y extrae/fusiona los NetCDF del ZIP resultante.
    La API v2 de CDS separa variables instant (u10, v10, t2m, d2m) y
    acumuladas (tp) en archivos distintos dentro de un ZIP.
    """
    tmp_path = output_path.with_suffix(".tmp")
    client.retrieve("reanalysis-era5-single-levels", request_params, str(tmp_path))

    with open(tmp_path, "rb") as f:
        magic = f.read(2)

    if magic == b"PK":
        with zipfile.ZipFile(tmp_path, "r") as zf:
            nc_names = [n for n in zf.namelist() if n.endswith(".nc")]
            print(f"  ZIP contiene: {nc_names}")

            if len(nc_names) == 1:
                extracted = zf.extract(nc_names[0], path=tmp_path.parent)
                Path(extracted).rename(output_path)
            else:
                # Múltiples NetCDF — extraer a temp dir y fusionar
                with tempfile.TemporaryDirectory() as tmpdir:
                    extracted_paths = [zf.extract(n, path=tmpdir) for n in nc_names]
                    datasets = [xr.open_dataset(p, engine="netcdf4") for p in extracted_paths]
                    merged = xr.merge(datasets)
                    merged.to_netcdf(str(output_path))
                    for ds in datasets:
                        ds.close()
        tmp_path.unlink()
    else:
        tmp_path.rename(output_path)


# Eliminar archivos previos incompletos (sin tp)
for req in REQUESTS:
    p = DATA_DIR / f"era5_{req['label']}.nc"
    if p.exists():
        p.unlink()
        print(f"Eliminado {p.name} (incompleto, faltaba tp)")

for req in REQUESTS:
    output_path = DATA_DIR / f"era5_{req['label']}.nc"

    print(f"\nSolicitando {req['label']}...")
    retrieve_era5(
        client,
        {
            "product_type": "reanalysis",
            "variable": VARIABLES,
            "year": req["year"],
            "month": req["month"],
            "day": DAYS,
            "time": TIMES,
            "area": ERA5_AREA,
            "format": "netcdf",
        },
        output_path,
    )

    size_kb = output_path.stat().st_size / 1024
    print(f"OK    {req['label']} → {output_path.name}  ({size_kb:.0f} KB)")

print("\nTodos los archivos ERA5 descargados.")

## 4. Verificación rápida

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

nc_files = sorted(DATA_DIR.glob("era5_*.nc"))

if nc_files:
    ds = xr.open_dataset(nc_files[0], engine="netcdf4")
    print(f"Archivo: {nc_files[0].name}")
    print(f"Variables: {list(ds.data_vars)}")
    print(f"Dimensiones: {dict(ds.dims)}")
    print(f"Fechas: {ds.valid_time.values[0]} → {ds.valid_time.values[-1]}")
    print(f"\nResumen t2m:")
    print(f"  Media: {float(ds['t2m'].mean()) - 273.15:.1f} °C")
    print(f"  Mín:   {float(ds['t2m'].min()) - 273.15:.1f} °C")
    print(f"  Máx:   {float(ds['t2m'].max()) - 273.15:.1f} °C")

    t2m_mean = ds["t2m"].mean(dim=["latitude", "longitude"]) - 273.15
    time_dim = ds.valid_time.values

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time_dim, t2m_mean.values, color="#e07b00", linewidth=1.5)
    ax.set_ylabel("Temperatura (°C)")
    ax.set_title(f"ERA5 — Temperatura media a 2m sobre AOI ({nc_files[0].stem})")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    out = Path("..") / "results" / f"era5_t2m_{nc_files[0].stem}.png"
    plt.savefig(out, dpi=150)
    plt.show()
    print(f"\nGráfico guardado en {out}")
    ds.close()
else:
    print("No se encontraron archivos ERA5.")

---
**Phase 1 completa.**

Siguiente paso: `Phase 2 — Preprocessing` (reproyección, cloud masking, índices espectrales, patch generation).